<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import Counter
import numpy as np
import pandas as pd


dataset = [
    [5.1, 3.5, 1.4, 0.2, "Iris-setosa"],
    [4.9, 3.0, 1.4, 0.2, "Iris-setosa"],
    [4.7, 3.2, 1.3, 0.2, "Iris-setosa"],
    [4.6, 3.1, 1.5, 0.2, "Iris-setosa"],
    [5.0, 3.6, 1.4, 0.2, "Iris-setosa"],
    [5.4, 3.9, 1.7, 0.4, "Iris-setosa"],
    [7.0, 3.2, 4.7, 1.4, "Iris-versicolor"],
    [6.4, 3.2, 4.5, 1.5, "Iris-versicolor"],
    [6.9, 3.1, 4.9, 1.5, "Iris-versicolor"],
    [5.5, 2.3, 4.0, 1.3, "Iris-versicolor"],
    [6.5, 2.8, 4.6, 1.5, "Iris-versicolor"],
    [5.7, 2.8, 4.5, 1.3, "Iris-versicolor"],
    [6.3, 3.3, 6.0, 2.5, "Iris-virginica"],
    [5.8, 2.7, 5.1, 1.9, "Iris-virginica"],
    [7.1, 3.0, 5.9, 2.1, "Iris-virginica"],
    [6.3, 2.9, 5.6, 1.8, "Iris-virginica"],
    [6.5, 3.0, 5.8, 2.2, "Iris-virginica"],
    [7.6, 3.0, 6.6, 2.1, "Iris-virginica"],
]

df = pd.DataFrame(
    dataset,
    columns=["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm", "Species"],
)

X = df[["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]].values.astype(float)
y = df["Species"].values

mean = X.mean(axis=0)
std = np.where(X.std(axis=0) == 0, 1.0, X.std(axis=0))
X_scaled = (X - mean) / std


def knn_predict_one(X_train, y_train, x_query, k=3):
    dists = np.linalg.norm(X_train - x_query, axis=1)
    nearest_idx = np.argsort(dists)[:k]
    votes = Counter(y_train[nearest_idx])
    return votes.most_common(1)[0][0]


def knn_predict(X_train, y_train, X_test, k=3):
    return np.array([knn_predict_one(X_train, y_train, row, k=k) for row in X_test])


test_indices = [4, 5, 10, 11, 16, 17]
train_mask = np.ones(len(X_scaled), dtype=bool)
train_mask[test_indices] = False

X_train, y_train = X_scaled[train_mask], y[train_mask]
X_test, y_test = X_scaled[test_indices], y[test_indices]

k = 3
preds = knn_predict(X_train, y_train, X_test, k=k)
accuracy = np.mean(preds == y_test) * 100

print(f"k = {k} | Test Accuracy: {accuracy:.2f}%\n")
for idx, actual, pred in zip(test_indices, y_test, preds):
    row = X[idx]
    print(
        f"Row {idx + 1:2d} | SL={row[0]:.1f}, SW={row[1]:.1f}, PL={row[2]:.1f}, PW={row[3]:.1f} | "
        f"Actual={actual:15s} | Pred={pred:15s}"
    )

new_samples = np.array([
    [5.0, 3.4, 1.5, 0.2],
    [6.1, 2.9, 4.7, 1.4],
    [6.7, 3.1, 5.6, 2.4],
])
new_scaled = (new_samples - mean) / std
new_preds = knn_predict(X_train, y_train, new_scaled, k=k)

print("\nNew Sample Predictions:")
for i, (sample, pred) in enumerate(zip(new_samples, new_preds), start=1):
    print(
        f"New {i} | SL={sample[0]:.1f}, SW={sample[1]:.1f}, PL={sample[2]:.1f}, PW={sample[3]:.1f} | "
        f"Predicted Species={pred}"
    )

k = 3 | Test Accuracy: 50.00%

Row  5 | SL=5.0, SW=3.6, PL=1.4, PW=0.2 | Actual=Iris-setosa     | Pred=Iris-setosa    
Row  6 | SL=5.4, SW=3.9, PL=1.7, PW=0.4 | Actual=Iris-setosa     | Pred=Iris-setosa    
Row 11 | SL=6.5, SW=2.8, PL=4.6, PW=1.5 | Actual=Iris-versicolor | Pred=Iris-virginica 
Row 12 | SL=5.7, SW=2.8, PL=4.5, PW=1.3 | Actual=Iris-versicolor | Pred=Iris-virginica 
Row 17 | SL=6.5, SW=3.0, PL=5.8, PW=2.2 | Actual=Iris-virginica  | Pred=Iris-virginica 
Row 18 | SL=7.6, SW=3.0, PL=6.6, PW=2.1 | Actual=Iris-virginica  | Pred=Iris-versicolor

New Sample Predictions:
New 1 | SL=5.0, SW=3.4, PL=1.5, PW=0.2 | Predicted Species=Iris-setosa
New 2 | SL=6.1, SW=2.9, PL=4.7, PW=1.4 | Predicted Species=Iris-virginica
New 3 | SL=6.7, SW=3.1, PL=5.6, PW=2.4 | Predicted Species=Iris-virginica
